<a href="https://colab.research.google.com/github/rasheed-hammad/machine-learning-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rasheed-hammad/machine-learning-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# One row = one content item/page for one client on one report date (a page-day). I will use the March 2026 mid-panel window, from `2026-03-01` to `2026-03-31`. This historical performance window will be used to build signals for ranking pages for content review and refresh decisions.


In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_Token")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [ ]:
from huggingface_hub import login

login(token=HF_TOKEN)

print("Connected to Hugging Face successfully.")

Connected to Hugging Face successfully.


In [ ]:
!pip -q install duckdb huggingface_hub

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Number of files:", len(files))
print("\nFirst few files:")
for f in files[:20]:
    print(f)

Number of files: 24

First few files:
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parqu

In [ ]:
import duckdb
from huggingface_hub import hf_hub_download

# Download the March 2026 warehouse partition
march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("March 2026 file downloaded:")
print(march_file)

March 2026 file downloaded:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [ ]:
con = duckdb.connect()

columns = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{march_file}')
""").fetchdf()

display(columns)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#2. Fields: feature / label / context / excluded

# Features:gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, and ga4_engaged_sessions. These are historical performance signals that can be used to prioritize content for review.

# Label / proxy: trend_direction, trend_pct, and is_declining_label. These represent the decline outcome/proxy and must not be used as model features.

# Context: report_date, client_hash_id, content_hash_id, and month. These fields are used for grouping, joining, splitting, or identifying the content item and client, rather than as predictive features.

# Excluded: gsc_sum_position, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, traffic-source fields, AI-source fields, and scroll_events. These are excluded from the initial five-feature scoring frame because they are metadata, availability indicators, or outside the selected feature set.


In [ ]:
# Verify the fields selected for the data contract

selected_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

print("Selected features:")
for feature in selected_features:
    print("-", feature)

print("\nAll selected features exist in the March 2026 table:")
available_columns = set(
    con.execute(f"""
        DESCRIBE SELECT *
        FROM read_parquet('{march_file}')
    """).fetchdf()["column_name"]
)

for feature in selected_features:
    print(feature, "->", feature in available_columns)

Selected features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions

All selected features exist in the March 2026 table:
gsc_impressions -> True
gsc_clicks -> True
gsc_avg_position -> True
ga4_pageviews -> True
ga4_engaged_sessions -> True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the page-day grain
grain_check = con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet('{march_file}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchdf()

display(grain_check)

print("Duplicate page-days found:", len(grain_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


Duplicate page-days found: 0


In [ ]:
# Verify row count and basic coverage

count_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet('{march_file}')
""").fetchdf()

display(count_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,clients,content_items,earliest_date,latest_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [ ]:
# Check missing values in the five selected features

missing_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,

        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_gsc_impressions,
        SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) AS missing_gsc_clicks,
        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS missing_gsc_avg_position,
        SUM(CASE WHEN ga4_pageviews IS NULL THEN 1 ELSE 0 END) AS missing_ga4_pageviews,
        SUM(CASE WHEN ga4_engaged_sessions IS NULL THEN 1 ELSE 0 END) AS missing_ga4_engaged_sessions

    FROM read_parquet('{march_file}')
""").fetchdf()

display(missing_check)

,total_rows,missing_gsc_impressions,missing_gsc_clicks,missing_gsc_avg_position,missing_ga4_pageviews,missing_ga4_engaged_sessions
0,9841378,0.0,0.0,6230317.0,3018741.0,3018741.0


In [ ]:
# Check GA4 availability against missing GA4 values

ga4_check = con.execute(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS rows,
        SUM(CASE WHEN ga4_pageviews IS NULL THEN 1 ELSE 0 END) AS missing_pageviews,
        SUM(CASE WHEN ga4_engaged_sessions IS NULL THEN 1 ELSE 0 END) AS missing_engaged_sessions
    FROM read_parquet('{march_file}')
    GROUP BY ga4_data_available
    ORDER BY ga4_data_available
""").fetchdf()

display(ga4_check)

,ga4_data_available,rows,missing_pageviews,missing_engaged_sessions
0,False,6408671,0.0,0.0
1,True,413966,0.0,0.0
2,<NA>,3018741,3018741.0,3018741.0


In [ ]:
# Check GSC availability against missing average position

gsc_check = con.execute(f"""
    SELECT
        gsc_data_available,
        COUNT(*) AS rows,
        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS missing_avg_position
    FROM read_parquet('{march_file}')
    GROUP BY gsc_data_available
    ORDER BY gsc_data_available
""").fetchdf()

display(gsc_check)

,gsc_data_available,rows,missing_avg_position
0,False,6230317,6230317.0
1,True,3611061,0.0


### 3. Verify it with queries

The March 2026 slice contains **9,841,378 page-day rows**, covering **55 clients** and **331,437 content items** from **2026-03-01 to 2026-03-31**.

The grain check found **0 duplicate combinations** of `report_date`, `client_hash_id`, and `content_hash_id`, supporting the page-day grain.

For the selected features, `gsc_impressions` and `gsc_clicks` have no NULL values. `gsc_avg_position` has **6,230,317 NULL values**, all associated with rows where `gsc_data_available` is `False`. The remaining **3,611,061** rows have GSC data available and no missing `gsc_avg_position`.

The GA4 fields `ga4_pageviews` and `ga4_engaged_sessions` each have **3,018,741 NULL values**. These occur in rows where `ga4_data_available` is NULL. Rows where `ga4_data_available` is `False` contain zero-filled GA4 values, while rows where it is `True` have populated GA4 values.

These results show that missingness is patterned by data availability rather than appearing uniformly random.



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4. Data limits

This warehouse cannot establish that a page will decline in the future because the available decline signal is a proxy based on observed data rather than a confirmed future outcome.

Client and content history is unbalanced, so not every client or content item has the same amount of historical data available. GSC availability also differs across rows: when GSC is unavailable, `gsc_avg_position` is missing.

GA4 availability also differs across the panel. In the March 2026 slice, some rows have GA4 unavailable with zero-filled metrics, while another group has NULL GA4 availability and NULL GA4 metrics. Therefore, GA4 fields should only be used when their availability is explicitly handled.

The March 2026 analysis is a single monthly slice and should be treated as measured and directional rather than assumed to generalize to every client or future period.

The resulting analysis should be used as **decision support for prioritizing content review**, not as a guarantee that a selected page will recover after a refresh.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
